In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## Notebook setup

This notebook reads the raw Gaspar source from `data/raw/` and writes the processed workbook to `data/processed/`.
The next cell resolves the project root whether the notebook is launched from the repository root or from `src/Preprocessing/`.


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GASPAR_RAW_PATH = RAW_DIR / "catnat_gaspar.csv"
GASPAR_OUTPUT_PATH = PROCESSED_DIR / "Gaspar_2015_2024.xlsx"

RAW_DIR, PROCESSED_DIR

(WindowsPath('D:/M2_MoSEF/DataCollection/data/raw'),
 WindowsPath('D:/M2_MoSEF/DataCollection/data/processed'))

## Load the raw Gaspar source

In [3]:
Gaspar_dataset = pd.read_csv(GASPAR_RAW_PATH, sep=";")

In [4]:
Gaspar_dataset

,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin,dat_pub_arrete,dat_pub_jo,dat_maj
0,BUDD8750027A,06120,Saint-Étienne-de-Tinée,GLT,Glissement de Terrain,1985-01-01,1986-12-31,1987-05-20,1987-05-24,2022-05-20 15:53:47.090
1,BUDD8750038A,2B050,Calvi,ICB,Inondations et/ou Coulées de Boue,1987-01-14,1987-01-15,1987-06-24,1987-07-10,2022-05-20 15:53:47.111
2,BUDD8750038A,30006,Aimargues,ICB,Inondations et/ou Coulées de Boue,1987-02-11,1987-02-13,1987-06-24,1987-07-10,2022-05-20 15:53:47.138
3,BUDD8750038A,30020,Aubord,ICB,Inondations et/ou Coulées de Boue,1987-02-11,1987-02-13,1987-06-24,1987-07-10,2022-05-20 15:53:47.138
4,BUDD8750038A,30033,Beauvoisin,ICB,Inondations et/ou Coulées de Boue,1987-02-11,1987-02-13,1987-06-24,1987-07-10,2022-05-20 15:53:47.138
...,...,...,...,...,...,...,...,...,...,...
260794,IOME2219351,12013,Aubin,ICB,Inondations et/ou Coulées de Boue,2022-06-04 02:00:00,2022-06-04 02:00:00,2022-07-09 02:00:00,2022-07-22 02:00:00,2022-09-30 02:00:00
260795,INTE2217496A,03110,Espinasse-Vozelle,ICB,Inondations et/ou Coulées de Boue,2022-06-03 02:00:00,2022-06-05 02:00:00,2022-06-15 02:00:00,2022-07-02 02:00:00,2022-09-30 02:00:00
260796,IOME2219351,53145,Marigné-Peuton,ICB,Inondations et/ou Coulées de Boue,2022-05-20 02:00:00,2022-05-20 02:00:00,2022-07-09 02:00:00,2022-07-22 02:00:00,2022-09-30 02:00:00
260797,INTE2217496A,22210,Ploubazlanec,ICB,Inondations et/ou Coulées de Boue,2022-06-03 02:00:00,2022-06-05 02:00:00,2022-06-15 02:00:00,2022-07-02 02:00:00,2022-09-30 02:00:00


## Convert date columns to datetime

In [5]:
Gaspar_dataset['dat_deb'] = pd.to_datetime(Gaspar_dataset['dat_deb'], format='mixed')

In [6]:
Gaspar_dataset['dat_fin'] = pd.to_datetime(Gaspar_dataset['dat_fin'], format='mixed')

## Keep the 2015-2024 comparison period used against the JRC tables

In [7]:
Gaspar_dataset_2015_2024 = Gaspar_dataset.loc[(Gaspar_dataset['dat_deb'] > '2015-01-01') & (Gaspar_dataset['dat_fin'] < '2024-12-31')]

## Review the hazard labels present in the filtered period

In [8]:
Gaspar_dataset_2015_2024['lib_risque_jo'].unique()

array(['Avalanche', 'Inondations et/ou Coulées de Boue',
       'Mouvement de Terrain',
       "Chocs Mécaniques liés à l'action des Vagues", 'Secousse Sismique',
       'Sécheresse', 'Inondations Remontée Nappe', 'Vents Cycloniques'],
      dtype=object)

## Keep flood-related hazards comparable with the JRC scope

We keep the main flood categories plus wave-action mechanical shocks, which can represent coastal flood impacts relevant for comparison.


In [9]:
FloodRiskLabels = [
    "Inondations et/ou Coul\u00e9es de Boue",
    "Inondations Remont\u00e9e Nappe",
    "Chocs M\u00e9caniques li\u00e9s \u00e0 l'action des Vagues",
]
Gaspar_dataset_2015_2024_Inondations = Gaspar_dataset_2015_2024[
    Gaspar_dataset_2015_2024["lib_risque_jo"].isin(FloodRiskLabels)
]

In [10]:
Gaspar_dataset_2015_2024_Inondations

,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin,dat_pub_arrete,dat_pub_jo,dat_maj
33194,INTE1507293A,2B246,La Porta,ICB,Inondations et/ou Coulées de Boue,2015-02-16 00:00:00,2015-02-17 00:00:00,2015-03-27,2015-03-31,2022-05-24 11:49:06.658
33210,INTE1507293A,62105,Belle-et-Houllefort,ICB,Inondations et/ou Coulées de Boue,2015-01-18 00:00:00,2015-01-18 00:00:00,2015-03-27,2015-03-31,2022-05-24 11:49:06.747
33211,INTE1507293A,62105,Belle-et-Houllefort,ICB,Inondations et/ou Coulées de Boue,2015-01-18 00:00:00,2015-01-18 00:00:00,2015-03-27,2015-03-31,2022-05-24 11:49:06.747
33212,INTE1507293A,62235,Condette,ICB,Inondations et/ou Coulées de Boue,2015-01-17 00:00:00,2015-01-18 00:00:00,2015-03-27,2015-03-31,2022-05-24 11:49:06.739
33214,INTE1507293A,62506,Licques,ICB,Inondations et/ou Coulées de Boue,2015-01-18 00:00:00,2015-01-18 00:00:00,2015-03-27,2015-03-31,2022-05-24 11:49:06.747
...,...,...,...,...,...,...,...,...,...,...
260794,IOME2219351,12013,Aubin,ICB,Inondations et/ou Coulées de Boue,2022-06-04 02:00:00,2022-06-04 02:00:00,2022-07-09 02:00:00,2022-07-22 02:00:00,2022-09-30 02:00:00
260795,INTE2217496A,03110,Espinasse-Vozelle,ICB,Inondations et/ou Coulées de Boue,2022-06-03 02:00:00,2022-06-05 02:00:00,2022-06-15 02:00:00,2022-07-02 02:00:00,2022-09-30 02:00:00
260796,IOME2219351,53145,Marigné-Peuton,ICB,Inondations et/ou Coulées de Boue,2022-05-20 02:00:00,2022-05-20 02:00:00,2022-07-09 02:00:00,2022-07-22 02:00:00,2022-09-30 02:00:00
260797,INTE2217496A,22210,Ploubazlanec,ICB,Inondations et/ou Coulées de Boue,2022-06-03 02:00:00,2022-06-05 02:00:00,2022-06-15 02:00:00,2022-07-02 02:00:00,2022-09-30 02:00:00


## Keep only the columns needed for downstream comparison

In [11]:
ColumnsToKeep = ['cod_nat_catnat', 'cod_commune', 'lib_commune', 'num_risque_jo', 'lib_risque_jo', 'dat_deb', 'dat_fin']
Gaspar_dataset_2015_2024_Inondations_Clean = Gaspar_dataset_2015_2024_Inondations.loc[:, ColumnsToKeep]

## Count duplicate rows before removing them

In [12]:
dup_mask = Gaspar_dataset_2015_2024_Inondations_Clean.duplicated(subset=ColumnsToKeep, keep='first')
n_duplicates = dup_mask.sum()
print(n_duplicates)

899


## Drop duplicate commune-event rows

In [13]:
Gaspar_dataset_2015_2024_Inondations_Clean = Gaspar_dataset_2015_2024_Inondations_Clean.drop_duplicates(subset=ColumnsToKeep).reset_index(drop=True)

In [14]:
Gaspar_dataset_2015_2024_Inondations_Clean

,cod_nat_catnat,cod_commune,lib_commune,num_risque_jo,lib_risque_jo,dat_deb,dat_fin
0,INTE1507293A,2B246,La Porta,ICB,Inondations et/ou Coulées de Boue,2015-02-16 00:00:00,2015-02-17 00:00:00
1,INTE1507293A,62105,Belle-et-Houllefort,ICB,Inondations et/ou Coulées de Boue,2015-01-18 00:00:00,2015-01-18 00:00:00
2,INTE1507293A,62235,Condette,ICB,Inondations et/ou Coulées de Boue,2015-01-17 00:00:00,2015-01-18 00:00:00
3,INTE1507293A,62506,Licques,ICB,Inondations et/ou Coulées de Boue,2015-01-18 00:00:00,2015-01-18 00:00:00
4,INTE1507293A,62643,Outreau,ICB,Inondations et/ou Coulées de Boue,2015-01-17 00:00:00,2015-01-18 00:00:00
...,...,...,...,...,...,...,...
19299,IOME2219351,12013,Aubin,ICB,Inondations et/ou Coulées de Boue,2022-06-04 02:00:00,2022-06-04 02:00:00
19300,INTE2217496A,03110,Espinasse-Vozelle,ICB,Inondations et/ou Coulées de Boue,2022-06-03 02:00:00,2022-06-05 02:00:00
19301,IOME2219351,53145,Marigné-Peuton,ICB,Inondations et/ou Coulées de Boue,2022-05-20 02:00:00,2022-05-20 02:00:00
19302,INTE2217496A,22210,Ploubazlanec,ICB,Inondations et/ou Coulées de Boue,2022-06-03 02:00:00,2022-06-05 02:00:00


## Build the small description sheet saved in the Excel workbook

In [15]:
dictionary = {
    'Sheet Name': ['Gaspar20152024FloodsClean', 'Gaspar20152024Floods', 'Gaspar20152024Full'],
    'Description': [
        'Gaspar flood-related events from 2015 to 2024, excluding duplicates',
        'Gaspar flood-related events from 2015 to 2024, including duplicates',
        'Gaspar all hazard events from 2015 to 2024'
    ]
}
Description = pd.DataFrame(dictionary)

## Save the processed workbook to `data/processed/`

In [16]:
with pd.ExcelWriter(GASPAR_OUTPUT_PATH) as writer:
    Gaspar_dataset_2015_2024_Inondations_Clean.to_excel(writer, sheet_name='Gaspar20152024FloodsClean', index=False)
    Gaspar_dataset_2015_2024_Inondations.to_excel(writer, sheet_name='Gaspar20152024Floods', index=False)
    Gaspar_dataset_2015_2024.to_excel(writer, sheet_name='Gaspar20152024Full', index=False)
    Description.to_excel(writer, sheet_name='Description Dictionary', index=False)

GASPAR_OUTPUT_PATH

WindowsPath('D:/M2_MoSEF/DataCollection/data/processed/Gaspar_2015_2024.xlsx')